# Aria Release & Production Operations Guide

**Complete guide for managing releases, production deployments, incident response, and operational runbooks.**

Learn how to safely deploy Aria to production and operate it at scale.


## Release Process

### Release Cycle

```
Development (Main Branch)
    ↓
1. Feature Complete
   - Code implemented
   - Tests passing
   - Documentation updated
    ↓
2. Release Candidate (v1.2.0-rc.1)
   - Create release branch
   - Final testing
   - Security review
    ↓
3. Release (v1.2.0)
   - Tag main branch
   - Create GitHub release
   - Build artifacts
    ↓
4. Staging Deployment
   - Deploy to staging environment
   - Run smoke tests
   - Monitor for 24 hours
    ↓
5. Production Deployment
   - Deploy to production
   - Monitor metrics
   - Ready for rollback
```

### Versioning Strategy

Use **Semantic Versioning**: `MAJOR.MINOR.PATCH`

```
v1.2.3
│ │ │
│ │ └─ PATCH: Bug fixes (v1.2.3 → v1.2.4)
│ └─── MINOR: New features (v1.2.0 → v1.3.0)
└───── MAJOR: Breaking changes (v1.0.0 → v2.0.0)

Examples:
- v1.0.0: Initial release
- v1.1.0: Add semantic memory feature
- v1.1.1: Fix chat provider fallback bug
- v2.0.0: Redesign Aria character system (breaking)
```

### Release Checklist

```markdown
## v1.2.0 Release Checklist

### Code Readiness

- [ ] All PRs merged to main
- [ ] Tests passing: `python scripts/test_runner.py --all`
- [ ] No critical bugs open
- [ ] Code reviewed

### Documentation

- [ ] CHANGELOG.md updated
- [ ] README.md updated if needed
- [ ] API docs updated
- [ ] Migration guide (if breaking)
- [ ] Release notes written

### Security

- [ ] Secrets scan passed
- [ ] Dependency audit clean
- [ ] Security review completed
- [ ] No known vulnerabilities

### Testing

- [ ] Unit tests: 100% pass
- [ ] Integration tests: 100% pass
- [ ] E2E tests: 100% pass
- [ ] Performance tests: no regression
- [ ] Smoke tests: all passing

### Deployment

- [ ] Deployment plan reviewed
- [ ] Rollback plan documented
- [ ] Monitoring setup verified
- [ ] On-call rotation scheduled

### Approval

- [ ] Tech lead approval ✓
- [ ] Product manager approval ✓
- [ ] Security team approval ✓
- [ ] Release manager approval ✓
```

### Creating a Release

```bash
#!/bin/bash
# Create release v1.2.0

# 1. Create release branch
git checkout -b release/v1.2.0

# 2. Update version in code
# Update VERSION file, setup.py, package.json, etc.
echo "1.2.0" > VERSION
git add VERSION
git commit -m "Release: bump to v1.2.0"

# 3. Create release candidate tag
git tag -a v1.2.0-rc.1 -m "Release Candidate 1"

# 4. Push to GitHub
git push origin release/v1.2.0 --tags

# 5. After final testing, create production tag
git tag -a v1.2.0 -m "Production Release"
git push origin v1.2.0

# 6. Create GitHub Release
gh release create v1.2.0 \
  --title "Aria v1.2.0 - Semantic Memory & Improvements" \
  --notes-file RELEASE_NOTES_v1.2.0.md
```

---

## Production Deployment

### Pre-Deployment Verification

```bash
#!/bin/bash
# Verify deployment readiness

set -e

echo "=== Pre-Deployment Verification ==="

# 1. Check code quality
echo "[1/6] Running code quality checks..."
python scripts/test_runner.py --unit --coverage > /dev/null

# 2. Check dependencies
echo "[2/6] Checking dependencies..."
pip-audit --desc

# 3. Check configuration
echo "[3/6] Validating configuration..."
python scripts/fast_validate.py

# 4. Check database migrations
echo "[4/6] Testing database migrations..."
python scripts/test_migrations.py

# 5. Check secrets management
echo "[5/6] Verifying secrets are not in code..."
git diff --cached | grep -E "password|secret|key|token" && exit 1 || true

# 6. Get final approval
echo "[6/6] Ready for deployment"
read -p "Continue with deployment? (yes/no) " confirm
[[ "$confirm" == "yes" ]] || exit 1

echo "✓ All pre-deployment checks passed"
```

### Canary Deployment

**Deploy to 5% of users first:**

```bash
#!/bin/bash
# Canary deployment: route 5% of traffic to new version

# 1. Deploy new version to staging slot
az webapp deployment slot create \
  --name aria-web \
  --resource-group aria-rg \
  --slot staging

# Copy release artifacts to staging
az webapp deployment source config-zip \
  --resource-group aria-rg \
  --name aria-web \
  --slot staging \
  --src-path deployment.zip

# 2. Route 5% traffic to staging
az resource update \
  --name aria-web \
  --resource-group aria-rg \
  --set "siteConfig.trafficManagerEnabled=true" \
  --resource-type "Microsoft.Web/sites"

az webapp traffic-routing set \
  --name aria-web \
  --resource-group aria-rg \
  --distribution staging=5  # 5% to staging, 95% to production

# 3. Monitor for 1 hour
echo "Monitoring canary deployment for 1 hour..."
for i in {1..60}; do
  sleep 60
  error_rate=$(curl -s http://aria-web.azurewebsites.net/api/ai/status | jq .error_rate)
  echo "[$i/60] Error rate: $error_rate%"

  if (( $(echo "$error_rate > 1" | bc -l) )); then
    echo "ERROR RATE TOO HIGH! Rolling back..."
    az webapp traffic-routing set \
      --name aria-web \
      --resource-group aria-rg \
      --distribution production=100
    exit 1
  fi
done

# 4. If successful, route 100% to new version
echo "✓ Canary deployment successful"
az webapp deployment slot swap \
  --resource-group aria-rg \
  --name aria-web \
  --slot staging
```

### Blue-Green Deployment

**Keep two identical production environments:**

```bash
#!/bin/bash
# Blue-Green deployment

# Current: Blue (v1.1.0)
# Target: Green (v1.2.0)

# 1. Deploy to Green environment
echo "Deploying v1.2.0 to Green..."
az webapp deployment source config-zip \
  --resource-group aria-rg \
  --name aria-green \
  --src-path deployment.zip

# 2. Run smoke tests on Green
echo "Running smoke tests on Green..."
pytest tests/smoke/all_endpoints.py --target aria-green || exit 1

# 3. Warm up Green (cache population, connection pools)
echo "Warming up Green..."
ab -c 100 -n 1000 https://aria-green.azurewebsites.net/api/ai/status

# 4. Switch traffic to Green
echo "Switching traffic to Green..."
az traffic-manager endpoint update \
  --profile-name aria-tm \
  --name green \
  --resource-group aria-rg \
  --type azureEndpoints \
  --endpoint-status Enabled

# 5. Monitor Green
echo "Monitoring Green for 30 minutes..."
for i in {1..30}; do
  sleep 60
  status=$(curl -s https://aria-green.azurewebsites.net/api/ai/status)
  echo "[$i/30] Status: $status"
done

# 6. If problems, switch back to Blue
# az traffic-manager endpoint update ... --endpoint-status Disabled
```

---

## Monitoring & Observability

### Key Metrics Dashboard

```
Aria Production Dashboard
├─ System Health
│  ├─ Availability: 99.95% ✓
│  ├─ Response Time (P95): 180ms ✓
│  └─ Error Rate: 0.02% ✓
│
├─ API Endpoints
│  ├─ /api/chat: 1,500 req/s, 150ms avg
│  ├─ /api/tts: 200 req/s, 500ms avg
│  └─ /api/quantum/*: 50 req/s, 2s avg
│
├─ Infrastructure
│  ├─ CPU Usage: 45% (target: <70%)
│  ├─ Memory: 62% (target: <80%)
│  └─ Disk: 72% (target: <85%)
│
└─ Database
   ├─ Query P95: 85ms (target: <100ms)
   ├─ Connections: 48/50 (target: <40)
   └─ Replication Lag: 200ms (target: <1s)
```

### Alert Configuration

```python
# Define critical alerts
ALERTS = {
    'high_error_rate': {
        'threshold': 0.05,  # 5%
        'duration': '5m',
        'severity': 'critical',
        'action': 'page_oncall'
    },
    'response_time_high': {
        'threshold': 2000,  # 2 seconds
        'duration': '10m',
        'severity': 'warning',
        'action': 'notify_slack'
    },
    'database_slow': {
        'threshold': 500,  # 500ms
        'duration': '5m',
        'severity': 'warning',
        'action': 'notify_slack'
    },
    'memory_high': {
        'threshold': 0.90,  # 90%
        'duration': '10m',
        'severity': 'warning',
        'action': 'auto_scale_up'
    },
    'disk_full_soon': {
        'threshold': 0.95,  # 95%
        'duration': '60m',
        'severity': 'critical',
        'action': 'page_oncall'
    }
}
```

---

## Incident Response

### Incident Classification

```
Severity 1 (Critical)
├─ Service completely down
├─ Data loss or corruption
├─ Security breach
└─ Revenue impacting

Severity 2 (High)
├─ Major feature broken
├─ Degraded performance (50%+)
├─ Affecting many users
└─ Need to respond within 15 min

Severity 3 (Medium)
├─ Minor feature broken
├─ Degraded performance (<50%)
├─ Affecting some users
└─ Need to respond within 1 hour

Severity 4 (Low)
├─ Cosmetic issue
├─ Performance issue (<10%)
├─ Single user affected
└─ Can wait for next release
```

### Incident Response Workflow

```
1. DETECT & ALERT
   ├─ Alert fires
   ├─ Page on-call engineer
   ├─ Create incident ticket
   └─ Slack notification

2. ASSESS & TRIAGE
   ├─ Check system status
   ├─ Review alerts & logs
   ├─ Assess impact (users, revenue, data)
   └─ Determine severity

3. COMMUNICATE
   ├─ Update status page
   ├─ Notify stakeholders
   ├─ Set expectations (ETA)
   └─ Update every 15 minutes

4. RESPOND
   ├─ Isolate problem
   ├─ Implement quick fix
   ├─ OR rollback to stable version
   └─ Verify fix works

5. RESOLVE
   ├─ Confirm issue resolved
   ├─ Monitor for recurrence
   ├─ Update status page
   └─ Close incident

6. POSTMORTEM
   ├─ Root cause analysis
   ├─ Identify preventive measures
   ├─ Document lessons learned
   └─ Schedule follow-up
```

### Common Incident Runbooks

**Runbook: API Returning 500 Errors**

```bash
#!/bin/bash
# Incident: API returning 500 errors

echo "=== Troubleshooting 500 Errors ==="

# 1. Check if service is running
echo "[1] Checking if service is running..."
az functionapp show --name aria-app --resource-group aria-rg | jq .state

# 2. Check recent logs
echo "[2] Checking recent logs..."
az webapp log tail --resource-group aria-rg --name aria-web | grep ERROR | tail -20

# 3. Check database connectivity
echo "[3] Testing database connectivity..."
sqlcmd -S aria-server.database.windows.net -U dbadmin -Q "SELECT 1"

# 4. Check external service dependencies
echo "[4] Checking external services..."
curl -s https://api.openai.com/v1/models | head -c 100

# 5. If still failing, restart service
echo "[5] Restarting service..."
az functionapp restart --name aria-app --resource-group aria-rg

# 6. Monitor
echo "[6] Monitoring error rate..."
for i in {1..10}; do
  sleep 30
  error_count=$(az webapp log show --name aria-web --resource-group aria-rg | grep ERROR | wc -l)
  echo "[$i/10] Error count: $error_count"
done
```

**Runbook: Database Performance Degradation**

```bash
#!/bin/bash
# Incident: Database queries very slow

echo "=== Database Performance Recovery ==="

# 1. Check current connections
echo "[1] Current connections:"
sqlcmd -S aria-server.database.windows.net -U dbadmin -Q "SELECT COUNT(*) as active_connections FROM sys.dm_exec_sessions WHERE session_id > 50"

# 2. Kill long-running queries
echo "[2] Killing long-running queries..."
sqlcmd -S aria-server.database.windows.net -U dbadmin -Q "
SELECT session_id FROM sys.dm_exec_requests
WHERE datediff(second, start_time, getdate()) > 60
"
# KILL [session_id]

# 3. Rebuild indexes
echo "[3] Rebuilding indexes..."
sqlcmd -S aria-server.database.windows.net -U dbadmin -Q "
ALTER INDEX ALL ON [aria_db].dbo.[messages] REBUILD
ALTER INDEX ALL ON [aria_db].dbo.[users] REBUILD
"

# 4. Update statistics
echo "[4] Updating statistics..."
sqlcmd -S aria-server.database.windows.net -U dbadmin -Q "
EXEC sp_updatestats
"

# 5. Scale up if CPU high
echo "[5] Checking CPU usage..."
db_cpu=$(az sql db show --server aria-server --name aria_db --resource-group aria-rg --query 'currentServiceObjectiveName')
echo "Current tier: $db_cpu"

# Scale up if needed
# az sql db update --server aria-server --name aria_db --service-objective S2 --resource-group aria-rg

echo "✓ Database recovery complete"
```

**Runbook: Out of Memory**

```bash
#!/bin/bash
# Incident: Service OOM killed

echo "=== Memory Pressure Recovery ==="

# 1. Check memory usage
echo "[1] Current memory usage:"
az monitor metrics list --resource aria-web --resource-group aria-rg --metric "MemoryUsagePercentage" | jq

# 2. Check for memory leaks
echo "[2] Analyzing memory trends..."
# Review Application Insights memory chart

# 3. Restart service (soft reset)
echo "[3] Restarting service..."
az functionapp restart --name aria-app --resource-group aria-rg

# 4. Scale up
echo "[4] Scaling up..."
az appservice plan update --name aria-web-plan --resource-group aria-rg --sku P2V2

# 5. Monitor
echo "[5] Monitoring memory..."
watch -n 10 'az monitor metrics list --resource aria-web --resource-group aria-rg --metric "MemoryUsagePercentage"'
```

---

## Change Management

### Change Log Template

```markdown
## Change Log

### [v1.2.0] - 2026-07-05

#### Added

- Semantic memory for improved chat context retention
- Quantum-augmented LLM responses
- Real-time Aria gesture feedback

#### Changed

- Improved chat provider fallback logic
- Optimized database queries (50% faster)
- Updated API documentation

#### Fixed

- Fixed null pointer in gesture handler
- Fixed memory leak in chat provider
- Fixed CORS issue with web UI

#### Security

- Added rate limiting to API endpoints
- Implemented MFA for admin access
- Scanned dependencies for vulnerabilities

#### Known Issues

- Quantum simulator has 5s overhead on first run
- LoRA adapter loading takes 30s (cold start)

#### Migration

- No breaking changes
- Optional: Update LoRA models for 10% accuracy improvement
```


## Production Runbook

### Daily Operations Checklist

```bash
#!/bin/bash
# Daily operations checklist

echo "=== Daily Operations Check ==="

# 1. Check system health
echo "✓ System health: OK"
curl -s http://aria-web.azurewebsites.net/api/ai/status | jq .

# 2. Check error rate
echo "✓ Error rate: $(az monitor metrics list --resource aria-web --resource-group aria-rg --metric 'Http5xx' --interval PT1H --aggregation Total | jq '.[0].timeseries[0].data[0].total // 0')%"

# 3. Check database
echo "✓ Database connections: $(sqlcmd -S aria-server.database.windows.net -U dbadmin -Q 'SELECT COUNT(*) FROM sys.dm_exec_sessions')"

# 4. Check storage
echo "✓ Storage usage: $(du -sh /var/lib/aria-data)"

# 5. Review recent logs
echo "✓ Recent errors:"
tail -20 /var/log/aria-app.log | grep ERROR

# 6. Check SSL certificate expiry
echo "✓ SSL certificate expires in: $(echo | openssl s_client -servername aria.app -connect aria.app:443 2>/dev/null | openssl x509 -noout -dates | grep notAfter)"
```

### Weekly Operations Checklist

```bash
#!/bin/bash
# Weekly operations checklist

echo "=== Weekly Operations Check ==="

# 1. Review metrics
echo "1. Review performance metrics for the week"
# Check Application Insights dashboard

# 2. Backup verification
echo "2. Verify daily backups are working"
az sql db backup list --server aria-server --name aria_db --resource-group aria-rg

# 3. Dependency updates
echo "3. Check for security updates"
pip-audit --desc

# 4. Capacity planning
echo "4. Review resource usage trends"
# Check month-over-month usage

# 5. Security scan
echo "5. Run security scan"
bandit -r shared/ -f json -o data_out/bandit_weekly.json

# 6. Update documentation
echo "6. Update runbooks and documentation"
```

### Quarterly Review

```bash
#!/bin/bash
# Quarterly review checklist

echo "=== Quarterly Review ==="

# 1. Architecture review
echo "1. Review architecture decisions"
echo "   - Are we still using optimal services?"
echo "   - Any tech debt to address?"

# 2. Capacity planning
echo "2. Project 3-month capacity needs"
echo "   - User growth"
echo "   - Feature expansion"
echo "   - Resource scaling"

# 3. Disaster recovery testing
echo "3. Test disaster recovery procedures"
echo "   - Database restore from backup"
echo "   - Service failover"
echo "   - Data integrity verification"

# 4. Security audit
echo "4. Conduct security audit"
echo "   - Penetration testing"
echo "   - Compliance review"
echo "   - Access control audit"

# 5. Cost optimization
echo "5. Analyze costs and optimize"
echo "   - Reserved instances"
echo "   - Unused services"
echo "   - Performance optimization"
```

---

## Operational Excellence

### Key Principles

```
1. Visibility
   Instrument everything
   Monitor, measure, alert

2. Automation
   Eliminate manual toil
   Standardize deployments
   Consistent environments

3. Reliability
   Test everything
   Graceful degradation
   Rapid recovery

4. Continuous Learning
   Blameless postmortems
   Document lessons
   Improve continuously

5. Shared Responsibility
   Dev owns their code in prod
   Ops enables dev autonomy
   Together we run prod
```

### On-Call Responsibilities

**When on-call, you are responsible for:**

1. **Respond to alerts** (within 5 minutes)
2. **Triage incidents** (within 15 minutes)
3. **Communicate status** (every 15 minutes)
4. **Resolve issues** (target SLA: 1 hour)
5. **Post-incident review** (within 24 hours)

**You have authority to:**

- Restart services
- Scale up resources
- Rollback deployments
- Temporarily disable features
- Execute approved runbooks

**Escalation path:**

1. On-call engineer (you)
2. Engineering manager
3. VP of Engineering
4. CTO (for critical incidents)
